### Описание docker файла

**image** - замороженный снимок системы, в котором установлены Ubuntu, Python, CUDA и vLLM.  

**container** - экземпляр этого образа, запущенный и работающий.

**docker compose** - инструмент, написания и запуска команд.  

**NVIDIA Container Toolkit** - позволяет Docker контейнерам «видеть» **видеокарты** хост-машины.   

1. **На хост-машине** устанавливаем драйверы NVIDIA.  

2. **В Docker** прописывается установка утилиты `nvidia-container-toolkit`.  

3. При запуске контейнера с флагом `--gpus all` (или через `resources` в Compose), этот инструмент пробрасывает ядра CUDA и драйверы внутрь изолированного контейнера.
  
Пример:

In [ ]:
image: nvcr.io/nvidia/pytorch:24.01-py3          # Образ с CUDA
container_name: deepseek-v4-flash-0731-nospec    # Название контейнера
network_mode: host               # Заставляет контейнер отказаться от собственной изолированной виртуальной сети и использовать сетевую систему хоста
privileged: true                 # Дает контейнеру полные root-права на уровне хост-системы (доступ к ядру, жестким дискам, сети и любым устройствам).
shm_size: "32gb"                 # Объем виртуальной памяти /dev/shm, выделяемый контейнеру для быстрого обмена данными между процессами (IPC) в обход медленного диска.
gpus: all                        # Предоставляет контейнеру доступ к видеокартам (GPU) хоста для ускорения вычислений (например, для нейросетей или майнинга).
environment:
  - CUDA_VISIBLE_DEVICES=0,1,2,3 # Переменная окружения, назначает первые четыре графических процессора (GPU) в системе доступными

---

#### Конструкция `CUDA_VISIBLE_DEVICES=0,1,2,3`:

- **Фиксация порядка (Index Mapping):** Порядок GPU для операционной системы и для библиотек (PyTorch, TensorFlow) может не совпадать. Явный запуск гарантирует, что нулевой индекс в коде точно сошлется на физическую карту `0`.  

- **Защита от чужих процессов:** Если скрипт запускается в кластере или на общем сервере (где другие пользователи могут динамически занимать карты), эта переменная жестко привязывает код к конкретным слотам.  

- **Поведение библиотек по умолчанию:** Некоторые фреймворки при инициализации пытаются занять вообще все видимые устройства. Перестраховка в команде запуска исключает непредвиденный выбор встроенной графики или других ускорителей.  

- **Шаблонизация кода:** При необходимости быстро поменять `0,1,2,3` на `0,2` без изменения логики самого Python-кода.

---

#### Разделяемая память `shm_size` в ML и инференсе  
Отвечает за то, чтобы центральный процессор (CPU) и оперативная память (RAM) успевали «кормить» видеокарту данными без задержек и ошибок.

1. **Многопроцессный Data Loading (PyTorch DataLoader):**  
    При обучении моделей PyTorch использует класс `DataLoader` с параметром `num_workers > 0`. Главный процесс создает дочерние воркеры для параллельной загрузки и аугментации картинок/текстов. Чтобы передать тяжелые батчи тензоров обратно в главный процесс, PyTorch копирует их в Shared Memory (`/dev/shm`). Если памяти там меньше, чем размер батча, процесс моментально падает с **`Bus error`**.

2. **Тензорный параллелизм и распределенное обучение (Distributed ML):**  
    При работе с большими языковыми моделями (LLM) или тяжелыми CV-сетями на нескольких GPU (фреймворки вроде _DeepSpeed_, _Megatron-LM_, _vLLM_, _Ray_) процессы на одной машине постоянно обмениваются градиентами и весами. Использование разделяемой памяти позволяет им коммуницировать с минимальной задержкой.
3. **Высокопроизводительный инференс (Triton Inference Server, TGI, vLLM):**  
    Современные сервера инференса (например, _NVIDIA Triton_ или _HuggingFace TGI_) используют IPC через Shared Memory для сверхбыстрой передачи входных данных (картинок, эмбеддингов, аудио) от веб-сервера (API-прослойки) к вычислительному ядру, работающему с GPU.

Если контейнеру не хватает разделяемой памяти, вы увидите одну из следующих ошибок: 
- RuntimeError: DataLoader worker (pid X) is killed by signal: Bus error.  
- ERROR: Unexpected bus error encountered in worker. This might be caused by the container running out of shared memory (shm).  
- Контейнер неожиданно завершается с кодом ошибки 135 (что соответствует сигналу SIGBUS).

---

#### Флаги командной строки 

`--tool-call-parser=deepseek_v4` (Парсеры инструментов / Tool Call Parsers) — это специальные программные модули, которые переводят скрытые рассуждения нейросети в структурированный формат (обычно JSON), когда модель решает вызвать внешнюю функцию (например, запустить калькулятор, сделать запрос в базу данных или вызвать API погоды).  

>Когда вы просите модель: «Узнай погоду в Москве и пришли ответ», модель генерирует текст. Парсер подхватывает этот текст, вычленяет из него команду типа {"name": "get_weather", "arguments": {"location": "Moscow"}} и передает ее вашей системе.  

`--reasoning-parser=deepseek_v4` («внутренний монолог» / Chain of Thought) — модель внутри себя пишет черновик кода, затем сама же «читает» его глазами и симулирует работу компилятора: «Так, если я передам сюда массив, то на шаге 5 возникнет утечка памяти. Нужно переписать». Только после этих виртуальных проверок она выдает итоговый вариант.  

>Скорость и качество написания кода у таких моделей в разы выше, но они всё равно физически не запускают двоичный файл на процессоре.  

Когда Docker запускает этот контейнер, он склеивает entrypoint (точку входа) и command (команду) в одну длинную строчку терминала. На физическом уровне внутри сервера выполняется обычная консольная команда:  
`vllm serve /model --host=0.0.0.0 --port=8000 --served-model-name=deepseek-v4-flash-0731 --tool-call-parser=deepseek_v4
`

---

#### Рекомендуемые настройки разделяемой памяти

🏋️‍♂️ Обучение (Training)

_Здесь `/dev/shm` критически важен для `num_workers > 0` в PyTorch DataLoader, когда процессоры параллельно готовят и передают батчи на GPU._

|Сценарий обучения|Рекомендуемый `shm_size`|Почему это важно|
|---|---|---|
|**Классическое ML** (Scikit-learn, XGBoost, таблицы)|**Дефолтные 64 МБ – 1 ГБ**|Обучение идет в один поток или данные лежат целиком в обычной RAM.|
|**Среднее CV** (YOLO, ResNet, небольшие батчи)|**2 ГБ – 8 ГБ**|Достаточно для параллельной загрузки картинок стандартного размера (например, 640x640).|
|**Тяжелое CV** (Много воркеров, FullHD, сегментация)|**16 ГБ – 64 ГБ**|Картинки высокого разрешения и аугментация «на лету» мгновенно забивают мелкий SHM.|
|**Видео-модели** (3D-CNN, Video Transformers)|**32 ГБ – 128 ГБ**|Батчи видео весят в десятки раз больше фото. Маленький SHM вызовет `Bus error`.|
|**NLP и Аудио** (BERT, Whisper, генерация речи)|**8 ГБ – 32 ГБ**|Токены весят мало, но SHM нужен для сборки батчей с динамическим паддингом.|
|**Обучение LLM / Reinforcement Learning** (Deepspeed, Ray)|**Равен RAM хоста** (или `--ipc=host`)|При работе Multi-GPU (DDP) или симуляторов в RL процессы постоянно обмениваются весами через SHM.|

🚀 Инференс (Inference)

_Здесь `/dev/shm` нужен для быстрой передачи тензоров между API-сервером (например, FastAPI) и движком инференса, а также для параллельной обработки очередей запросов._

|Сценарий инференса|Рекомендуемый `shm_size`|Почему это важно|
|---|---|---|
|**Простые модели** (Таблицы, простые классификаторы текстовые)|**Дефолтные 64 МБ – 512 МБ**|Данные передаются в один поток, SHM почти не используется.|
|**CV инференс** (Распознавание лиц, детекция объектов поштучно)|**1 ГБ – 4 ГБ**|Обычно инференс картинок идет с `batch_size=1`. Память нужна только на IPC-передачу кадра.|
|**Видео-аналитика** (Стриминг с камер, обработка видеопотока)|**8 ГБ – 32 ГБ**|Декодированные кадры из видеопотоков должны мгновенно передаваться в модель.|
|**LLM Инференс** (vLLM, Triton, Hugging Face TGI)|**16 ГБ – 64 ГБ**|**Критично!** Используется для алгоритмов разбиения тензоров (Tensor Parallelism) и управления KV-кэшем.|

💡 Главное правило

Если вы сомневаетесь, какой размер выставить, или запускаете **Multi-GPU (несколько видеокарт)** для любой задачи — всегда пишите флаг **`--ipc=host`** (в docker run) или **`ipc: host`** (в docker-compose). Это полностью сотрет границы между хостом и контейнером в плане общей памяти и застрахует от любых ошибок падения по памяти.